# Isolated UF policy BGE fine-tuning
Run from the repository root with the project `.venv` kernel. This notebook calls the separate modules in `src/finetuning`; it never opens the app's `policies` collection for writing. Synthetic answers are drafts, not UF policy advice.


In [ ]:
from src.finetuning.common import ARTIFACTS, ROOT
from src.finetuning.manifest import build
print(ROOT, ARTIFACTS)
print(build())


## Generate a pilot
Set `RUN_GENERATION=True` only after configuring `OPENAI_API_KEY`. Generation uses two concurrent network requests by default, requests JSON mode with up to 1,200 output tokens, and retries malformed/empty output once with a 2,000-token cap. A quote that is not copied verbatim triggers one stricter retry for that chunk. It resumes from the existing draft JSONL, selects train and development policies only, and excludes questions from the old answer-comparison workbook. The worker count is configurable (1–8); increase cautiously if the API gateway allows it. Failures are printed and saved in `artifacts/qa_failures.v1.jsonl`; rerunning retries failed chunks. Review every draft: verify its answer and exact quote against the official page, then change `review_status` to `approved` or `rejected` and add `reviewed_by` and `review_note`. Keep the JSONL valid.


In [ ]:
RUN_GENERATION = False
if RUN_GENERATION:
    from src.finetuning.qa import generate
    print(generate(ARTIFACTS, count=250, workers=2))
else:
    print('Generation disabled. Set RUN_GENERATION=True after reading the review instructions.')


## Pilot training
Human approval is mandatory. The first run is capped at ten steps and saves to a new directory. The old model and database stay untouched. On macOS, device `auto` selects MPS when available; use CPU if MPS is unstable.


In [ ]:
RUN_TRAINING = False
CHECKPOINT = ARTIFACTS / 'checkpoints/pilot_v1'
if RUN_TRAINING:
    from src.finetuning.train import train
    print(train(ARTIFACTS, CHECKPOINT, device='auto', epochs=1, batch_size=8, lr=2e-5, seed=42, max_steps=10))
else:
    print('Training disabled until the QA drafts have been reviewed.')


## Independent retrieval test and index
Write human-authored `artifacts/human_test.v1.jsonl` using **test-policy** chunk IDs only. Each line needs `question`, `positive_chunk_id`, `policy_id`, `question_type`, and `reviewed_by`. Lock it once with `python -m src.finetuning.evaluate --lock-test`, then run evaluation with `--checkpoint`. The index builder requires a non-regressing fine-tuned result and writes only to `artifacts/chroma_finetuned_v1`. See the local README for exact commands. Do not substitute the old 49 exposed evaluation questions for this test.
